In [ ]:
#@title 按這裡開始（先按 ▶）
print('✅ W08 出發！本週目標：做一個真的有人會用的分類器，再用 OpenCV 看它之前發生了什麼')
print('這一週有一段要離開瀏覽器，用電腦上的 Python 讀攝影機；裝不起來就用本檔的 Colab 備援。')

# W08　分類器專題＋用 OpenCV 看前處理

**先存副本**：`檔案 → 在雲端硬碟中儲存副本`，檔名 `AI導論_W08_學號_姓名`。

## 任務一：選題目與定類別

三選一：**垃圾分類**、**手勢辨識**、**有沒有戴口罩**。

| 題目 | 類別怎麼分 | 拍照要注意 |
|---|---|---|
| 垃圾分類 | 紙類、寶特瓶、鋁罐、都不是 | 每種找兩三個不同品牌的實物來拍 |
| 手勢辨識 | 剪刀、石頭、布、都不是 | 左右手都要拍，遠近各拍一些 |
| 有沒有戴口罩 | 有戴、沒戴、都不是 | 找不同同學入鏡，換兩種背景 |
| 共通原則 | 每一類的張數要接近 | 每類 50 到 60 張，不要超過 200 張 |

**一定要加一類「都不是」**，鏡頭空著時才不會硬判成其中一類。

## 任務二：用電腦攝影機拍照與訓練

1. **先拍測試用的**：每類另外拍 10 張，訓練時絕對不放進去
2. **再拍訓練用的**：每類 50 到 60 張，換角度、光線與背景
3. **按 Train Model**：用上週學到的設定，訓練時不要切分頁
4. **自己先測一次**：用剛剛留下來的測試畫面測 20 次
5. **不夠好就先補資料**：先補資料再調參數，這是上週的結論

**匯出與分享**：`Export Model → Tensorflow.js → Upload my model`（給別組交叉測試，
上傳要等 1 到 2 分鐘），連結用畫面上的 **Copy 按鈕**複製，不要用滑鼠框選。
進階任務要的是 `Export Model → Tensorflow → Keras → Download`（`keras_model.h5`）。

## 任務三：自己寫出混淆矩陣

去別組的連結，**每一類各測 10 次**，把數字填進 `rows`。

**寫對了會看到**：一張 3×3 的表格，以及 `交叉測試正確率 = 0.833`（用範例數字時）。
對角線是判對的，其他格子就是最需要補資料的地方。

In [ ]:
import pandas as pd

names = ["紙類", "寶特瓶", "鋁罐"]
rows  = [[9, 1, 0],           # 真的是紙類，被判成三類各幾次
         [2, 7, 1],           # 真的是寶特瓶
         [0, 1, 9]]           # 真的是鋁罐 ← 換成你的實測

cm = pd.DataFrame(rows, index=names, columns=names)
display(cm)

right = ____                  # ← 自己寫：對角線加總
total = ____                  # ← 自己寫：全部次數加總
print("交叉測試正確率 =", round(right / total, 3))

## 任務三之二：哪一類最容易被判錯

每一類自己測對的比例，就是那一類的**召回率**。

**寫對了會看到**：`[0.9, 0.7, 0.9]` 與一張長條圖。
最矮的那一根，就是回去要補拍照片的那一類。

In [ ]:
import matplotlib.pyplot as plt

per = []
for i in range(len(names)):
    per.append(____)          # ← 自己寫：第 i 類測對的比例
print([round(a, 2) for a in per])

plt.bar(range(len(names)), per)
plt.xticks(range(len(names)), ["c1", "c2", "c3"])
plt.ylabel("recall"); plt.ylim(0, 1)
plt.savefig("w08_recall.png", dpi=150)
plt.show()

## 任務四之前：先確認 OpenCV 跑得起來

任務四要**離開瀏覽器**，用電腦上的 Python 直接讀攝影機。

1. 先在命令提示字元試一次：`python --version`
   （沒有回應就改用 Anaconda Prompt，或直接走下面的 Colab 備援）
2. 裝套件：`python -m pip install opencv-python`
   （全班同時裝會塞，請照座位分兩批，各等一分鐘再開始）
3. **Colab 備援**：不裝任何東西也能做同樣三件事，改成讀示範影片、用 `cv2_imshow` 顯示
4. 先確認攝影機**沒有被別的程式佔用**（關掉視訊軟體與相機 App）

## 任務四：OpenCV 讀攝影機做三件事（本機 Python）

程式檔在本 repo 的 `ai-intro-pc/scripts/webcam_gray_edge.py`，
下載到電腦後在命令提示字元執行 `python webcam_gray_edge.py`。
**下面兩行要自己補**（檔案裡也是留空的）：

```python
import cv2

cap = cv2.VideoCapture(0)        # 0 是內建攝影機
while True:
    ok, frame = cap.read()
    if not ok:
        break
    gray = ____                  # ← 自己寫：轉成灰階
    edge = cv2.Canny(gray, 100, 200)
    small = cv2.resize(frame, ____)  # ← 自己寫：縮成 224
    cv2.imshow("gray", gray)
    cv2.imshow("edge", edge)
    if cv2.waitKey(1) == 27:     # 按 Esc 離開
        break
cap.release()
cv2.destroyAllWindows()
```

**會看到**：兩個視窗（灰階、邊緣）。灰階、邊緣、縮小，
正是模型看到圖之前做的三件事。按 `Esc`（腳本也接受 `q`）離開。

### 備援前的準備：確認 `sample.mp4` 在不在

老師會把示範影片放在課程雲端硬碟。**這一格是自救用的**：
教室下載不到影片時，會自己生一段 30 影格的示範影片，觀念完全一樣。

In [ ]:
# ←投影片未含，執行所需：沒有老師的 sample.mp4 就自己產生一段示範影片
import os, cv2, numpy as np

if not os.path.exists('sample.mp4'):
    w, h = 640, 480
    vw = cv2.VideoWriter('sample.mp4',
                         cv2.VideoWriter_fourcc(*'mp4v'), 15, (w, h))
    for t in range(30):
        f = np.full((h, w, 3), 40, dtype=np.uint8)
        cv2.rectangle(f, (100 + t * 5, 120), (300 + t * 5, 320),
                      (0, 180, 220), -1)
        cv2.circle(f, (480, 240), 70, (220, 220, 220), 3)
        vw.write(f)
    vw.release()

print('sample.mp4 準備好了嗎？', os.path.exists('sample.mp4'))

## 任務四備援：Colab 讀影片檔做同樣三件事

教室電腦裝不了套件時改用這一格，**觀念完全一樣**。
Colab 沒有視窗系統，所以要用 `cv2_imshow` 而不是 `imshow`。

**會看到**：`True (480, 640, 3)`，接著兩張圖 —— 灰階與邊緣。

In [ ]:
import cv2
from google.colab.patches import cv2_imshow

cap = cv2.VideoCapture("sample.mp4")  # 雲端硬碟的示範影片
ok, frame = cap.read()
print(ok, frame.shape)                # True (480, 640, 3)

gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
edge  = cv2.Canny(gray, 100, 200)
small = cv2.resize(frame, (224, 224))

cv2_imshow(gray)
cv2_imshow(edge)
cap.release()

## 進階任務：把模型接上 OpenCV 即時推論（本機 Python）

先做完 W05 的 Keras 匯出，把 `keras_model.h5` 與 `labels.txt` 放在同一個資料夾，
存成 `live.py` 再執行 `python live.py`。

```python
import cv2, numpy as np
from tensorflow.keras.models import load_model

model  = load_model("keras_model.h5", compile=False)
labels = open("labels.txt", encoding="utf-8").read().splitlines()
cap = cv2.VideoCapture(0)
while True:
    ok, frame = cap.read()
    if not ok:
        break
    img = cv2.cvtColor(cv2.resize(frame, (224, 224)),
                       cv2.COLOR_BGR2RGB)
    x = (img.astype("float32") / 127.5) - 1
    p = model.predict(x.reshape(1, 224, 224, 3), verbose=0)[0]
    print(labels[p.argmax()], round(float(p.max()), 2))
    cv2.imshow("cam", frame)
    if cv2.waitKey(1) == 27:
        break
```

跑完記得補 `cap.release()` 與 `cv2.destroyAllWindows()`。

## 繳交與常見狀況

要交：筆記本連結、模型分享連結、`w08_recall.png`、混淆矩陣的實測數字。

| 狀況 | 原因 | 怎麼處理 |
|---|---|---|
| 攝影機畫面全黑 | 被別的程式佔用 | 關掉視訊軟體與相機 App 再重整分頁 |
| 瀏覽器不給用攝影機 | 權限被拒絕過 | 點網址列鎖頭 → 攝影機 → 允許 |
| 上傳模型卡住不動 | 全班同時上傳塞車 | 等一分鐘重試，或分兩批錯開上傳 |
| pip 裝了卻 import 失敗 | 裝到別的 Python | 改用 `python -m pip install` 再裝一次 |
| `cv2.imshow` 在 Colab 當掉 | Colab 沒有視窗 | 改用 `cv2_imshow`，或回本機那一格 |
| 別組模型幾乎全判錯 | 場景和他拍的不同 | 這正是本週重點，把情況記錄下來 |
| 重開機後檔案不見 | 存在 C 槽不是雲端 | 模型與筆記本一律放自己的雲端硬碟 |
| 下載模型檔被擋 | 瀏覽器擋住下載 | 點網址列提示，允許這個網站下載 |

### 延伸挑戰（A 到 C 越後面越難）

- **A 改參數再跑**：把 Canny 的 `100, 200` 改成 `30, 90` 再跑一次。邊緣變多還是變少？為什麼？
- **B 換一批資料**：針對最矮的那一根補拍 20 張再訓練一次。召回率長條圖有沒有變平？
- **C 說出為什麼**：為什麼別人來測，分數就掉下來？用「訓練資料跟真實情況不一樣」解釋。

### 下週期中考：範圍與題型

| 考什麼 | 範圍與題型 | 配分與提醒 |
|---|---|---|
| 選擇題 | 第 1 到 8 週的觀念判斷，共 20 題 | 40%　每題 2 分，不倒扣 |
| 名詞解釋 | 過擬合、監督式、卷積、epoch 之類，4 題 | 20%　定義＋用途＋一個例子 |
| 程式判讀 | 迴圈與 if、pandas 五個指令的輸出 | 25%　過程寫出來就有部分分數 |
| 簡答與流程 | 給情境，說明怎麼切資料、看哪個指標 | 15%　講得通就給分 |
| 不會考的部分 | 工具的操作步驟、按鈕在畫面哪個位置 | 考的是觀念與判讀 |
| 要特別看熟 | 第 5、6、7 週的表格與評估指標 | 往年失分最多的三週 |
| 考試形式 | 在電腦教室紙筆作答 60 分鐘，螢幕關閉 | 可帶手寫 A4 單面小抄一張 |

---

<details>
<summary><b>參考解</b>（真的卡住再打開，先自己試滿 10 分鐘）</summary>

**任務三**

```python
right = sum(rows[i][i] for i in range(len(names)))   # 對角線加總 = 25
total = sum(sum(r) for r in rows)                    # 全部次數 = 30
```

**任務三之二**

```python
    per.append(rows[i][i] / sum(rows[i]))            # 第 i 類測對的比例
```

**任務四（本機 `webcam_gray_edge.py`）**

```python
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)   # 轉灰階
    small = cv2.resize(frame, (224, 224))            # 縮成 224x224
```

</details>